# Model Training & Evaluation
## Personal Cognitive Load & Focus Monitoring System

This notebook trains and evaluates models for cognitive load prediction.

### Objectives:
1. Load processed data
2. Split data into train/test sets
3. Train models (rule-based or lightweight ML)
4. Evaluate model performance
5. Log experiments with MLflow
6. Register best model


In [1]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path
import sys
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow
import mlflow.sklearn
from datetime import datetime

# Add project root and src to path
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
for path in [PROJECT_ROOT, PROJECT_ROOT / "src"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

# Set paths
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

# Initialize MLflow
MLFLOW_DB = PROJECT_ROOT / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")
mlflow.set_experiment("personal-cognitive-load-monitor")

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("MLflow experiment:", mlflow.get_experiment_by_name("personal-cognitive-load-monitor"))


MLflow tracking URI: sqlite:////home/pnmynh/Documents/personal-cognitive-load-monitor/mlflow.db
MLflow experiment: <Experiment: artifact_location='/home/pnmynh/Documents/personal-cognitive-load-monitor/notebooks/mlruns/1', creation_time=1792806893483, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1792806893483, lifecycle_stage='active', name='personal-cognitive-load-monitor', tags={}, trace_location=None, workspace='default'>


In [2]:
# Load processed data
X = pd.read_csv(DATA_PROCESSED / "features.csv")
y = pd.read_csv(DATA_PROCESSED / "target.csv").squeeze()

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nTarget distribution:")
print(y.value_counts())


Features shape: (500, 6)
Target shape: (500,)

Target distribution:
cognitive_load_level
MEDIUM    305
LOW       120
HIGH       75
Name: count, dtype: int64


In [3]:
# Match scripts/train_model.py: deterministic chronological 80/20 isolation.
split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTrain target distribution:")
print(y_train.value_counts())
print(f"\nTest target distribution:")
print(y_test.value_counts())


Train set: 400 samples
Test set: 100 samples

Train target distribution:
cognitive_load_level
MEDIUM    245
LOW        96
HIGH       59
Name: count, dtype: int64

Test target distribution:
cognitive_load_level
MEDIUM    60
LOW       24
HIGH      16
Name: count, dtype: int64


In [4]:
# Rule baseline: expected to be perfect because it generated synthetic_proxy labels.
from preprocessing_api.service import calculate_cognitive_load
from preprocessing_api.schemas import PredictionRequest

def predict_with_service(X):
    """Predict using the service function."""
    predictions = []
    for _, row in X.iterrows():
        request = PredictionRequest(
            focus_minutes=int(row["focus_minutes"]),
            distraction_minutes=int(row["distraction_minutes"]),
            tasks_due=int(row["tasks_due"]),
            hours_to_deadline=float(row["hours_to_deadline"])
        )
        pred = calculate_cognitive_load(request)
        predictions.append(pred.value)
    return np.array(predictions)

# Test the service
y_pred_service = predict_with_service(X_test)
accuracy_service = accuracy_score(y_test, y_pred_service)

print("Rule-based Service Performance:")
print(f"Accuracy: {accuracy_service:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_service))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_service))


Rule-based Service Performance:
Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

        HIGH       1.00      1.00      1.00        16
         LOW       1.00      1.00      1.00        24
      MEDIUM       1.00      1.00      1.00        60

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100


Confusion Matrix:
[[16  0  0]
 [ 0 24  0]
 [ 0  0 60]]


In [5]:
# Option 2: Train a simple ML model (e.g., Random Forest)
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Encode labels
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

# Train model
with mlflow.start_run(run_name=f"random_forest_{datetime.now().strftime('%Y%m%d_%H%M%S')}"):
    # Train
    model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
    model.fit(X_train, y_train_encoded)
    
    # Predict
    y_pred = model.predict(X_test)
    y_pred_labels = le.inverse_transform(y_pred)
    
    # Evaluate
    accuracy = accuracy_score(y_test_encoded, y_pred)
    
    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    
    # Log model
    mlflow.sklearn.log_model(model, "model")
    
    print(f"Random Forest Accuracy: {accuracy:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_labels))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred_labels))
    
    # Register model
    model_name = "cognitive-load-classifier"
    mlflow.sklearn.log_model(
        model,
        "model",
        registered_model_name=model_name
    )
    print(f"\nModel registered as: {model_name}")


2026/10/23 21:31:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/10/23 21:31:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/10/23 21:31:25 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/10/23 21:31:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/10/23 21:31:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest Accuracy: 0.9200

Classification Report:
              precision    recall  f1-score   support

        HIGH       0.93      0.88      0.90        16
         LOW       0.91      0.88      0.89        24
      MEDIUM       0.92      0.95      0.93        60

    accuracy                           0.92       100
   macro avg       0.92      0.90      0.91       100
weighted avg       0.92      0.92      0.92       100


Confusion Matrix:
[[14  0  2]
 [ 0 21  3]
 [ 1  2 57]]


2026/10/23 21:31:29 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.



Model registered as: cognitive-load-classifier


Registered model 'cognitive-load-classifier' already exists. Creating a new version of this model...
Created version '4' of model 'cognitive-load-classifier'.


## Summary

### Interpretation
- The generating-rule baseline is expected to score 1.0; this exposes label circularity, not real-world validity.
- Candidate metrics measure fidelity to synthetic proxy rules only. They do not measure cognition, health, or productivity.

### Next Steps:
- Load one exact registered artifact and verify train-serving feature parity.
- Collect separately consented `self_report` labels before any real-world validity claim.
